# UNSW-NB15 Data Analysis
This notebook processes the UNSW-NB15 dataset from Google Drive to count rows, aggregate source IP statistics, and resolve IP addresses to country names.

# UNSW-NB15 Data Analysis
This notebook processes the UNSW-NB15 dataset from Google Drive to count rows, aggregate source IP statistics, and resolve IP addresses to country names.

## Step 2: Connect Drive
Mount Google Drive to access the dataset files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 3: Load the data and count rows
This loads only the required columns to prevent out-of-memory errors.

In [ ]:
import pandas as pd
import numpy as np
import glob
import ipaddress

base = '/content/drive/MyDrive/unsw/'
idx  = [0, 1, 2, 3, 4, 5, 47, 48]
cols = ['srcip','sport','dstip','dsport','proto','state','attack_cat','Label']

dfs = []
for f in sorted(glob.glob(base + 'UNSW-NB15_[1-4].csv')):
    d = pd.read_csv(f, header=None, usecols=idx,
                    dtype={1: str, 3: str}, low_memory=False)
    d.columns = cols
    d['srcip'] = d['srcip'].str.strip()
    print(f.split('/')[-1], len(d), 'rows')
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
print('TOTAL rows:', len(df))
print('Distinct source IPs:', df['srcip'].nunique())
print(df['Label'].value_counts())

UNSW-NB15_1.csv 700001 rows
UNSW-NB15_2.csv 700001 rows
UNSW-NB15_3.csv 700001 rows
UNSW-NB15_4.csv 440044 rows
TOTAL rows: 2540047
Distinct source IPs: 43
Label
0    2218764
1     321283
Name: count, dtype: int64


## Step 4: Summarise per source IP

In [ ]:
src = (df.groupby('srcip')
         .agg(rows=('Label','size'), attacks=('Label','sum'))
         .sort_values('rows', ascending=False)
         .reset_index())
print(src.to_string())

              srcip    rows  attacks
0        59.166.0.4  197959        0
1        59.166.0.1  197680        0
2        59.166.0.5  197626        0
3        59.166.0.2  197550        0
4        59.166.0.0  197528        0
5        59.166.0.3  195953        0
6        59.166.0.9  190187        0
7        59.166.0.6  189419        0
8        59.166.0.8  189341        0
9        59.166.0.7  189059        0
10     175.45.176.1  127998   117908
11     175.45.176.3  118673   106418
12     175.45.176.0   82999    74279
13   149.171.126.18   81680        0
14   149.171.126.15   44883        0
15   149.171.126.14   41047        0
16     175.45.176.2   32026    22678
17   149.171.126.10   30365        0
18   149.171.126.12   16082        0
19       10.40.85.1    4018        0
20      10.40.182.1    3984        0
21      10.40.182.6    3492        0
22      10.40.85.30    2138        0
23      10.40.182.3    2105        0
24      10.40.170.2    2094        0
25      10.40.85.10     793        0
2

## Step 5: Look up countries
We map the source IP addresses to geographic country data using binary search.

In [ ]:
geo = pd.read_csv(base + 'country_asn.csv')
geo = geo[~geo['start_ip'].str.contains(':')].copy()   # keep IPv4 only

def to_int(x):
    try: return int(ipaddress.IPv4Address(x))
    except: return -1

geo['s'] = geo['start_ip'].map(to_int)
geo['e'] = geo['end_ip'].map(to_int)
geo = geo.sort_values('s').reset_index(drop=True)
starts, ends = geo['s'].values, geo['e'].values

src['ip_int'] = src['srcip'].map(to_int)
pos = np.searchsorted(starts, src['ip_int'].values, side='right') - 1
safe = np.clip(pos, 0, None)
ok = (pos >= 0) & (src['ip_int'].values <= ends[safe])
src['country'] = np.where(ok, geo['country_name'].values[safe], 'Unresolved')

print(src[['srcip','rows','attacks','country']].to_string())
print()
print(src.groupby('country')[['rows','attacks']].sum()
         .sort_values('attacks', ascending=False))

              srcip    rows  attacks      country
0        59.166.0.4  197959        0        Japan
1        59.166.0.1  197680        0        Japan
2        59.166.0.5  197626        0        Japan
3        59.166.0.2  197550        0        Japan
4        59.166.0.0  197528        0        Japan
5        59.166.0.3  195953        0        Japan
6        59.166.0.9  190187        0        Japan
7        59.166.0.6  189419        0        Japan
8        59.166.0.8  189341        0        Japan
9        59.166.0.7  189059        0        Japan
10     175.45.176.1  127998   117908  North Korea
11     175.45.176.3  118673   106418  North Korea
12     175.45.176.0   82999    74279  North Korea
13   149.171.126.18   81680        0    Australia
14   149.171.126.15   44883        0    Australia
15   149.171.126.14   41047        0    Australia
16     175.45.176.2   32026    22678  North Korea
17   149.171.126.10   30365        0    Australia
18   149.171.126.12   16082        0    Australia
